### Use of Ollama open-source software uisng Ollama library

In [2]:
pip install ollama

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.14.1-py3-none-any.whl.metadata (133 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.50.1-cp311-cp311-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached pydantic-2.14.1-py3-none-any.whl (488 kB)
Using cached pydantic_core-2.50.1-cp311-cp311-win_amd64.whl (2.2 MB)
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import ollama

In [8]:
import ollama

# stream=True returns an iterator, needs to be iterated through
stream = ollama.generate(
    model="ministral-3:8b",
    prompt="What is the capital of India?",
    stream=True
)

# Iterate through each chunk and print the text
for chunk in stream:
    print(chunk['response'], end='', flush=True)

The capital of India is **New Delhi**. It serves as both the political and administrative center of the country.

In [9]:
# giving multiple input the model at once
inputs = [
    "What is the capital of India?",
    "What is the capital of France?",
    "What is the capital of Germany?"
]

for prompt in inputs:
    response = ollama.generate(model="ministral-3:8b", prompt=prompt)
    print(response['response'])

The capital of India is **New Delhi**. It is also one of the largest cities in the country and serves as the political and administrative center of the nation.
The capital of France is **Paris**.
The capital of Germany is **Berlin**.


### Tool Calling the Ollama model using Ollama library

In [10]:
import ollama 

# TOOL 1: Mock Database for Inventory
def check_inventory(product_name: str):
    """Checks the local database for stock levels of a specific product."""
    inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }
    product = product_name.lower()
    if product in inventory_db:
        return inventory_db[product]
    return {"error": "Product not found"}




# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price: float, years_as_customer: int):
    """Calculates a custom discount: 5% per year, capped at 25%."""
    discount_pct = min(years_as_customer * 0.05, 0.25)
    final_price = base_price * (1 - discount_pct)
    return {"final_price": final_price, "discount_applied": f"{discount_pct*100}%"}

In [11]:
# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [12]:
# 1. Define the tool schemas for the LLM
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'check_inventory',
            'description': 'Get stock and price for a product',
            'parameters': {
                'type': 'object',
                'properties': {
                    'product_name': {'type': 'string', 'description': 'Name of the product'}
                },
                'required': ['product_name'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate_loyalty_discount',
            'description': 'Calculate final price based on customer loyalty years',
            'parameters': {
                'type': 'object',
                'properties': {
                    'base_price': {'type': 'number', 'description': 'The original price'},
                    'years_as_customer': {'type': 'integer', 'description': 'How many years they have been a member'}
                },
                'required': ['base_price', 'years_as_customer'],
            },
        },
    }
]

In [13]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string',
      'description': 'Name of the product'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on customer loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number',
      'description': 'The original price'},
     'years_as_customer': {'type': 'integer',
      'description': 'How many years they have been a member'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [15]:
# 3. Execution Loop
messages = [{'role': 'user', 'content': 'Do we have laptops in stock? If so, what is the price for a customer who has been with us for 3 years?'}]

# First call: LLM decides which tools to use
response = ollama.chat(model='ministral-3:8b', messages=messages, tools=tools)

In [16]:
response

ChatResponse(model='ministral-3:8b', created_at='2026-10-24T07:53:02.9172754Z', done=True, done_reason='stop', total_duration=52756362300, load_duration=17191594000, prompt_eval_count=720, prompt_eval_duration=32807886000, eval_count=14, eval_duration=2702242000, message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))]), logprobs=None)

In [17]:
response["message"]

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))])

In [19]:
# Process tool calls
if response['message'].get('tool_calls'):
    for tool in response['message']['tool_calls']:
        function_to_call = available_functions[tool['function']['name']]
        args = tool['function']['arguments']
        
        # Execute local Python code
        result = function_to_call(**args)
        
        # Add the tool's output back to the conversation
        messages.append(response['message'])
        messages.append({'role': 'tool', 'content': str(result)})

# Final call: LLM summarizes the findings
final_response = ollama.chat(model='ministral-3:8b', messages=messages)
print(final_response['message']['content'])

Yes, we currently have laptops in stock.

For a customer who has been with us for **3 years**, the price would be **€1,080** (a 10% discount applied to the base price of €1,200).

Would you like to proceed with a purchase or need further details?
